# i.MX95 Debug Authentication
i.MX95 offers the Debug Authentication Protocol (DAP) as a mechanism to authenticate the debugger (an external entity) which has the credentials approved by the product manufacturer before granting the debug access to the device. This example demonstrates the process of establishing the Debug Authentication protocol.

The process could be split into the following steps:


## Introduction
The fundamental principles of debugging, which require access to the system state and system information, conflict with the principles of security, which require the restriction of access to assets. Thus, many products disable debug access completely before deploying the product. To address these challenges, the chip offers a debug authentication protocol as a mechanism to authenticate the debugger (an external entity) has the credentials approved by the product manufacturer before granting debug access to the device.
The debug authentication is a challenge-response scheme and assures that only the debugger in possession of the required debug credentials can successfully authenticate over the debug interface and access restricted parts of the device.


The protocol is divided into steps as described below:
1. The debugger initiates the Debug Mailbox message exchange by setting the CSW[RESYNCH_REQ] bit and CSW[CHIP_RESET_REQ] bit of DM-AP.
2. The debugger waits (minimum 30 ms) for the device to restart and enter the debug mailbox request handling loop.
3. The debugger sends the Debug Authentication Start command (command code 10h) to the device.
4. The device responds back with a Debug Authentication Challenge (DAC) packet based on the debug access rights preconfigured in the fuses, which are collectively referred as Device Credential Constraints Configuration (DCFG_CC). The response packet also contains a 32-byte random challenge vector.
5. The debugger responds to the challenge with a Debug Authentication Response (DAR) message by using an appropriate debug certificate, matching the device identifier in the DAC. The DAR packet contains the debug access permission certificate, also referred as Debug Credential (DC), and a cryptographic signature binding the DC and the challenge vector provided in the DAC.
6. The device on receiving the DAR, validates the contents by verifying the cryptographic signature of the message using the debugger's public key present in the embedded Debug Credential (DC). On successful validation of the DAR, the device enables access to the debug domains permitted in the DC.


![debug_authentication_flow](../img/debug_authentication_flow.png)


## 1. Prerequisites
- SPSDK is needed with examples extension. `pip install spsdk[examples]` (Please refer to the [installation](../../_knowledge_base/installation_guide.rst) documentation.)
- The device must be properly configured to run a signed application (example of running a signed application is here: [i.MX95 signed AHAB with U-BOOT](../../ahab/imx95/imx95_signed_ahab_uboot.ipynb))
- The device life cycle must be moved to the state OEM_CLOSED
- This example uses the i.MX95 FRDM board. This is an example board configuration with an external debugger. In our case we used a Segger J-Link probe connected over a standard 20 pin JTAG connector.

### 1.1 Let's prepare the environment


In [1]:
from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

WORKSPACE = "workspace/"  # change this to path to your workspace
INPUTS = "inputs/"
VERBOSITY = (
    "-v"  # verbosity of commands, might be -v or -vv for debug or blank for no additional info
)

DC_CONFIG = INPUTS + "dc_config.yaml"  # debug credential file configuration
DAT_CONFIG = INPUTS + "dat_config.yaml"  # debug authentication configuration
DC_FILE_PATH = WORKSPACE + "mx95_dc.bin"  # exported debug credential file

DEBUGGER_PROBE = "jlink"  # Segger J-Link probe

FAMILY = "mimx9596"

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


## 2. Generate RoT + Debug auth keys

First we need to generate the Root of Trust Keys (RoTKs)/Super Root Keys (SRKs) and the Debug Credential Key (DCK). Use the *nxpcrypto* app to generate secp384r1 keys (see [How-to-get-keys-using-nxpcrypto](../../crypto/keys.ipynb)). We will need by default 4 RoTKs and 1 DCK.

The notebook is using pre-generated keys just for the purposes of this example from the folder `_data/keys/ecc384` in the root of all SPSDK examples.

## 3. Generate debug credential file including its configuration file

First we need to prepare the configuration file for the `debug credential file` export. The template for its configuration can be created using the `nxpdebugmbox dat dc get-template` command. To simplify this example, we have already prepared that configuration, which can be found in the [./inputs/dc_config.yaml](./inputs/dc_config.yaml) file. Below, we'll compare the differences between the template and our customized example to highlight the additions we've made.

> **Note**: To get the UUID of the device, connect the debug probe into the connector and read the UUID by `nxpdebugmbox -f mimx9596 tool get-uuid`. Another way to get the UUID is to use the `nxpele get-info` command which communicates via the U-Boot serial line or fastboot.


In [2]:
# Get difference of template and user YAML configuration
YamlDiffWidget("inputs/dc_config.diffc").html

nxpdebugmbox dat dc get-template -f mimx9596 -o workspace/dc_config.yaml --force 
Creating workspace/dc_config.yaml template file.


Now we generate the dc file based on the yaml configuration.


In [3]:
%! nxpdebugmbox dat dc export -c $DC_CONFIG -o $DC_FILE_PATH --force

nxpdebugmbox dat dc export -c inputs/dc_config.yaml -o workspace/mx95_dc.bin --force 
Creating Debug credential file succeeded


## 4. Generate debug authentication configuration file

In advance we will prepare the configuration file for the debug authentication procedure itself. Let's begin again by creating a template configuration file using the `nxpdebugmbox dat get-template` command. To simplify this example, we have already prepared that configuration, which can be found in the [./inputs/dat_config.yaml](./inputs/dat_config.yaml) file. Below, we'll compare the differences between the template and our customized example.


In [4]:
# Get difference of template and user YAML configuration
YamlDiffWidget("inputs/dat_config.diffc").html

nxpdebugmbox dat get-template -f mimx9596 -o workspace/dat_config.yaml --force 
Creating workspace/dat_config.yaml template file.


## 5. Test debug authentication
Now we can call the authentication command for nxpdebugmbox. Since the board is in the OEM closed life cycle, as a validation test we will use the test-connection command from the `nxpdebugmbox` application. After the auth command is performed, the chip should enable the debug interface based on the settings in the CC_SOCU field in the debug credential file.


In [5]:
# Test to confirm that debug access does NOT work
%! nxpdebugmbox $VERBOSITY -i $DEBUGGER_PROBE mem-tool test-connection -f $FAMILY

nxpdebugmbox -v -i jlink mem-tool test-connection -f mimx9596 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851002903   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
  #   Gate                                                                                      Status      
------------------------------------------------------------------------------------------------------------
  0   Debug probe opened (PC is correctly connected to the debug probe hardware)                PASSED      
  1   Probe attached to target (debug probe is correctly connected to the board via SWD/JTAG)   PASSED      
  2   Debug Port IDR readable (target chip responds on the debug port)                          PASSED      
  3   DP power domains up / CDBGPWRUPACK+CSYSPWRUPACK (target power supply and clocks are on)   PASSED      
  4   Access Port IDR rea

SPSDKAppError: The test connection was not successful.


In [6]:
# Unlock the Debug Authentication
%! nxpdebugmbox $VERBOSITY -i $DEBUGGER_PROBE dat auth -c $DAT_CONFIG

nxpdebugmbox -v -i jlink dat auth -c inputs/dat_config.yaml 
INFO:spsdk.apps.nxpdebugmbox:Starting Debug Authentication
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851002903   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
INFO:spsdk.apps.nxpdebugmbox:DAC: 
Version                : Version 2.0
SOCC                   : 0x4D58005F
UUID                   : 0219319DA89C4F13A32655A0187B393A
CC_VU                  : 0
ROTID_rkh_revocation   : 00000000
ROTID_rkth_hash        : 459a16d16b4b8ffdb438de839480886e9ce3812b5443fa5856ceaa386590bb86
CC_soc_pinned          : 00FFFFF0
CC_soc_default         : 00000000
Challenge              : c23d88774d3f38095c1866280d81949ecedda28003861ff3540c7aabbecfb6b6

INFO:spsdk.apps.nxpdebugmbox:
============================================== DAC versus DC (Warning) =============================================

In [7]:
# Test to confirm that debug access works
%! nxpdebugmbox $VERBOSITY -i $DEBUGGER_PROBE mem-tool test-connection -f $FAMILY

nxpdebugmbox -v -i jlink mem-tool test-connection -f mimx9596 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851002903   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
INFO:spsdk.apps.nxpdebugmbox:Debug-access gating status:
  #   Gate                                                                                      Status      
------------------------------------------------------------------------------------------------------------
  0   Debug probe opened (PC is correctly connected to the debug probe hardware)                PASSED      
  1   Probe attached to target (debug probe is correctly connected to the board via SWD/JTAG)   PASSED      
  2   Debug Port IDR readable (target chip responds on the debug port)                          PASSED      
  3   DP power domains up / CDBGPWRUPACK+CSYSPWRUPACK (target power supply an